In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
completions_raw = pd.read_csv(
    r"..\data\raw\c2024_a.csv",
    dtype=str, 
    low_memory=False
)

print(completions_raw.shape)
completions_raw.head()

(307707, 64)


,UNITID,CIPCODE,MAJORNUM,AWLEVEL,XCTOTALT,CTOTALT,XCTOTALM,CTOTALM,XCTOTALW,CTOTALW,...,XCUNKNM,CUNKNM,XCUNKNW,CUNKNW,XCNRALT,CNRALT,XCNRALM,CNRALM,XCNRALW,CNRALW
0,100654,01.0999,1,5,R,12,R,2,R,10,...,Z,0,Z,0,Z,0,Z,0,Z,0
1,100654,01.1001,1,5,R,10,R,1,R,9,...,Z,0,Z,0,Z,0,Z,0,Z,0
2,100654,01.1001,1,7,R,7,R,1,R,6,...,Z,0,Z,0,R,2,R,1,R,1
3,100654,01.1001,1,17,R,6,R,3,R,3,...,R,0,R,2,R,3,R,2,R,1
4,100654,01.9999,1,5,R,2,R,1,R,1,...,Z,0,Z,0,Z,0,Z,0,Z,0


In [3]:
# Define the columns
cols = [
    "UNITID",
    "CIPCODE",        # cip6 format (xx.xxxx)
    "AWLEVEL",        # credential level
    "CTOTALT",        # total completions
    "CTOTALM",        # male
    "CTOTALW"        # female
]

completions = completions_raw[cols].copy()

Clean columns

In [4]:
# Unitid
completions["unitid_n"] = (
    completions["UNITID"]
    .astype(str)
    .str.strip()
)

In [5]:
# cip6 to cip4
completions["cip6"] = completions["CIPCODE"].astype(str).str.strip()
completions["cip6_n"] = completions["cip6"].str.replace(".", "", regex=False)
completions["cip4"] = completions["cip6_n"].str[:4]

In [6]:
# Drop cip 99 (a placeholder for programs that cannot be traditionally coded)
completions = completions[completions["cip4"] != "99"]

In [7]:
# AWLEVEL to credlev_n
completions['credlev_n'] = pd.to_numeric(completions["AWLEVEL"], errors="coerce").astype("Int64")

In [8]:
# Completions counts
for c in ["CTOTALT", "CTOTALM", "CTOTALW"]:
    completions[c] = pd.to_numeric(completions[c], errors="coerce").fillna(0).astype(int)

completions = completions.rename(columns={
    "CTOTALT": "completions",
    "CTOTALM": "completions_men",
    "CTOTALW": "completions_women"
})

In [9]:
# Add a year column
completions["year"] = 2024

In [10]:
# Create a dictionary for the credlev codes, limiting it to the ones I will be using
credlev_map = {
    1: [2, 3, 5],  # IPEDS codes for certificates
    2: [4],     # IPEDS codes for associates
    3: [6]         # IPEDS codes for bachelors
}

def map_credlevel(awlevel):
    for proj_level, ipeds_codes in credlev_map.items():
        if awlevel in ipeds_codes:
            return proj_level
    return np.nan

completions["credlev_proj"] = completions["credlev_n"].apply(map_credlevel)

In [11]:
# Aggregate: unitid × cip4 × credlev × year
completions_2024_agg = (
    completions
    .groupby(["unitid_n", "cip4", "credlev_proj", "year"], as_index=False)
    .agg({
        "completions": "sum",
        "completions_men": "sum",
        "completions_women": "sum"
    })
)

In [12]:
# Drop rows with 0 completions
completions_2024_agg = completions_2024_agg[
    (completions_2024_agg["completions"] > 0) |
    (completions_2024_agg["completions_men"] > 0) |
    (completions_2024_agg["completions_women"] > 0)
]

In [13]:
completions_2024_agg["credlev_proj"] = completions_2024_agg["credlev_proj"].astype(int)

In [14]:
# Name credlev_proj descriptions
cred_desc_map = {
    1: "Certificate",
    2: "Assoicate",
    3: "Bachelor"
}

completions_2024_agg["cred_desc"] = completions_2024_agg["credlev_proj"].map(cred_desc_map)

In [15]:
# Create composite key
completions_2024_agg["composite_key"] = (
    completions_2024_agg["unitid_n"] + "|" +
    completions_2024_agg["cip4"] + "|" +
    completions_2024_agg["credlev_proj"].astype(str)
)

In [16]:
# uniqueness
completions_2024_agg["composite_key"].is_unique

True

In [17]:
# null checks
completions_2024_agg.isna().sum()

unitid_n             0
cip4                 0
credlev_proj         0
year                 0
completions          0
completions_men      0
completions_women    0
cred_desc            0
composite_key        0
dtype: int64

In [18]:
# sanity check
completions_2024_agg.sample(10)

,unitid_n,cip4,credlev_proj,year,completions,completions_men,completions_women,cred_desc,composite_key
7827,112260,2602,1,2024,3,1,2,Certificate,112260|2602|1
108043,243115,1107,1,2024,21,15,6,Certificate,243115|1107|1
109552,363165,5126,1,2024,9,5,4,Certificate,363165|5126|1
95736,228501,4303,1,2024,2,2,0,Certificate,228501|4303|1
58624,185590,1611,1,2024,1,0,1,Certificate,185590|1611|1
2611,105145,1101,1,2024,10,5,5,Certificate,105145|1101|1
111708,417992,5139,1,2024,51,3,48,Certificate,417992|5139|1
3405,106245,2301,1,2024,23,4,19,Certificate,106245|2301|1
70231,199218,5123,1,2024,29,1,28,Certificate,199218|5123|1
1665,102580,1312,1,2024,1,1,0,Certificate,102580|1312|1


In [19]:
# Export
completions_2024_agg.to_csv(
    "completions_2024_model.csv",
    index=False,
    encoding="utf-8"
)